In [ ]:
import os
from typing import List, Dict

from dotenv import load_dotenv
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

In [ ]:
students: List[Dict] = [
    {
        "student_id": "22CS045",
        "name": "Dhanushya",
        "department": "Computer Science",
        "python": 85,
        "database": 72,
        "ai": 90,
        "web": 78,
    },
    {
        "student_id": "22CS046",
        "name": "Rahul",
        "department": "Computer Science",
        "python": 65,
        "database": 70,
        "ai": 68,
        "web": 72,
    },
    {
        "student_id": "22CS047",
        "name": "Priya",
        "department": "Information Technology",
        "python": 92,
        "database": 88,
        "ai": 95,
        "web": 90,
    },
    {
        "student_id": "22CS048",
        "name": "Arun",
        "department": "Information Technology",
        "python": 55,
        "database": 60,
        "ai": 58,
        "web": 62,
    },
    {
        "student_id": "22CS049",
        "name": "Meena",
        "department": "Computer Science",
        "python": 78,
        "database": 85,
        "ai": 80,
        "web": 88,
    },
]


In [ ]:
def find_student(student_id: str):
    for student in students:
        if student["student_id"] == student_id:
            return student

    return None

In [ ]:
@tool
def get_student_info(student_id: str) -> str:
    """
    Get the name and department of a student using their student ID.
    Use this tool when the user asks for a student's name or department.
    """

    student = find_student(student_id)

    if not student:
        return f"Student {student_id} not found."

    return (
        f"Name: {student['name']}\n"
        f"Department: {student['department']}"
    )

In [ ]:
@tool
def get_student_marks(student_id: str) -> str:
    """
    Get the Python, Database, AI, and Web marks of a student.
    Use this tool when the user asks about student marks.
    """

    student = find_student(student_id)

    if not student:
        return f"Student {student_id} not found."

    return (
        f"Python: {student['python']}\n"
        f"Database: {student['database']}\n"
        f"AI: {student['ai']}\n"
        f"Web: {student['web']}"
    )


In [ ]:
@tool
def calculator(expression: str) -> str:
    """
    Calculate a mathematical expression.
    Use this tool to calculate total marks or average marks.
    """

    try:
        result = eval(expression, {"__builtins__": {}}, {})

        return str(result)

    except Exception as e:
        return f"Calculation error: {str(e)}"

In [ ]:

@tool
def get_passing_rules() -> str:
    """
    Get the university passing requirements.
    Minimum overall average is 40%.
    Minimum mark in each subject is 35%.
    """

    return (
        "University Passing Rules:\n"
        "1. Minimum overall average: 40%\n"
        "2. Minimum mark in each subject: 35%"
    )



In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [ ]:
tools = [
    get_student_info,
    get_student_marks,
    calculator,
    get_passing_rules,
]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
You are a student information assistant.

You have access to student information and marks through tools.

Rules:

1. Use get_student_info when the user asks for a student's name or department.

2. Use get_student_marks when the user asks for marks.

3. Use calculator when you need to calculate total or average marks.

4. Use get_passing_rules when the user asks whether a student satisfies
   the university passing requirements.

5. You may call multiple tools when required.

6. Do not calculate totals or averages mentally.
   Always use the calculator tool.

7. For passing eligibility:
   - Get the student's marks.
   - Get the passing rules.
   - Calculate the total and average using the calculator.
   - Check every subject against the minimum mark requirement.
   - Then provide the final answer.

8. Do not call tools unnecessarily.

Answer clearly and concisely.
"""
)

In [ ]:
question = input("Enter the question")
response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        }
    )

print(response)